# SmolVLA の微調整 — 積み木のお片付け(ステージ2b: 同じ画面で対象を入れ替えたデモを足したデータ)
Google Drive の同期フォルダ(`PAI2026 Workspace`。自動で探す)のデータで、公開済みの `lerobot/smolvla_base` を微調整し、結果を同じ Drive に保存する。
**ランタイム → ランタイムのタイプを変更 → GPU(A100 または L4)** にしてから、上から順に実行する。
- ACT と同じ lerobot 0.3.3 系(コンテナ側の SmolVLA 用の環境と同じ版)を使う。ACT 用の `lerobot_src` は使わない。
- 命令文はデータの `tasks.jsonl`(英語)がそのまま入る。カメラは俯瞰1台、状態は5次元(32次元まで自動で埋められる)。

- ステージ2b: データは `tools\stage2_gen_WRITE.bat pair` で作り、Drive へ上げてある(名前は names.conf の S2PAIR。S2DATA の全デモ+同じ画面で対象を入れ替えた同数のデモ)。指示文は「put the {red|blue|green} block in the box」の3種。
- 出力先は新しい名前(`<データ名>_smolvla`)。既にあれば止まる(上書きしない)。


In [ ]:
# 1) Drive をつなぎ、同期フォルダを自動で探す(lerobot_dataset を含むフォルダ)
from google.colab import drive
drive.mount('/content/drive')
import os, glob, sys, shutil, subprocess, json, time
def sh(cmd):
    # コマンドを実行して出力をそのまま表示する。失敗したら止める
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end='')
    p.wait()
    assert p.returncode == 0, '失敗(終了コード %d): %s' % (p.returncode, cmd)
NAME = 's2_mix_1025'                                 # データ名(tools\names.conf の S2MIX と同じにする)
DRIVE_ROOT = None                                    # 自動で見つからないときだけ、ここに '/content/drive/MyDrive/…' を直接書く
if DRIVE_ROOT is None:
    hits = sorted(glob.glob('/content/drive/MyDrive/lerobot_dataset/'+NAME) + glob.glob('/content/drive/MyDrive/*/lerobot_dataset/'+NAME) + glob.glob('/content/drive/MyDrive/*/*/lerobot_dataset/'+NAME) + glob.glob('/content/drive/MyDrive/*/*/*/lerobot_dataset/'+NAME))
    print('見つかった候補:', hits)
    assert hits, 'lerobot_dataset/%s が Drive に見つからない。PCで xfer_WRITE.bat out data %s が終わっているか確認し、DRIVE_ROOT を直接書く' % (NAME, NAME)
    DRIVE_ROOT = os.path.dirname(os.path.dirname(hits[0]))
print('DRIVE_ROOT =', DRIVE_ROOT)

OUT = f'{DRIVE_ROOT}/checkpoints/train/{NAME}_smolvla'
assert not os.path.exists(OUT), '保存先が既にあります(上書きしません): ' + OUT + '。NAME を変えるか、先に別名へ退避する'
print('保存先(新規):', OUT)


In [ ]:
# 2) GPU の確認
sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
print(sys.version)


In [ ]:
# 3) lerobot 0.3.3(SmolVLA つき)を入れる(5〜10分)。※ 先に「ランタイム → セッションを再起動」しておくこと
sh('pip install -q torch==2.7.0 torchvision==0.22.0 torchcodec==0.4.0')
sh('pip install -q --only-binary=:all: "av>=14.2.0" datasets==3.6.0')
sh('pip install -q "lerobot[smolvla]==0.3.3"')
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy; print('SmolVLA import OK')


In [ ]:
# 4) データを Colab のローカルディスクへ(Drive のまま読むと遅い)。命令文とフレーム数も確認する
shutil.rmtree('/content/dataset', ignore_errors=True)
os.makedirs('/content/dataset')
shutil.copytree(f'{DRIVE_ROOT}/lerobot_dataset/{NAME}', f'/content/dataset/{NAME}')
d = f'/content/dataset/{NAME}/meta'
info = json.load(open(d + '/info.json'))
print('version', info.get('codebase_version'), 'fps', info.get('fps'), 'episodes', info.get('total_episodes'), 'frames', info.get('total_frames'))
for k, v in info['features'].items(): print('  ', k, v.get('dtype'), v.get('shape'))
print(open(d + '/tasks.jsonl').read())
assert info.get('total_episodes', 0) >= 100, 'エピソードが100件未満: データの生成・移送が途中の可能性'
ntask = len([l for l in open(d + '/tasks.jsonl') if l.strip()])
assert ntask >= 3, '指示文が %d 種しかありません(3種のはず)' % ntask


In [ ]:
# 6) 本番の学習。5000ステップごとに /content/smol_out へ保存する(Drive は symlink が使えないので、いったんローカルへ)
#    BATCH=32, STEPS=30000(デモが約2倍になったので1.5倍。A100 で 1ステップ約0.23秒 → 約115分)
BATCH, STEPS = 32, 30000
shutil.rmtree('/content/smol_out', ignore_errors=True)
t0 = time.time()
sh(f'python -m lerobot.scripts.train --policy.path=lerobot/smolvla_base --dataset.repo_id={NAME} --dataset.root=/content/dataset/{NAME} '
   f'--batch_size={BATCH} --steps={STEPS} --num_workers=4 --output_dir=/content/smol_out --job_name=smolvla_{NAME} --policy.device=cuda '
   f'--policy.push_to_hub=false --wandb.enable=false --log_freq=200 --save_freq=5000')
print('所要 %.1f 分' % ((time.time() - t0) / 60))


In [ ]:
# 7) Drive へ保存(モデルの重みと設定だけ。最後の1つ+途中の各ステップ)
os.makedirs(OUT, exist_ok=False)
for ck in sorted(glob.glob('/content/smol_out/checkpoints/*')):
    if os.path.islink(ck): continue
    src = ck + '/pretrained_model'
    if os.path.isdir(src):
        dst = f'{OUT}/{os.path.basename(ck)}'
        shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst); print('保存', dst)
last = sorted([c for c in glob.glob(OUT + '/0*')])[-1]
shutil.rmtree(f'{OUT}/final', ignore_errors=True); shutil.copytree(last, f'{OUT}/final')
sh(f'ls -la "{OUT}/final"; du -sh "{OUT}"')


## 終わったら
評価は PC 側の `stage2_eval.bat seen URL TOKEN`(学習配置の再現。約40分)→ `stage2_eval.bat vla URL TOKEN`(条件 C → A → B)で行う。推論サーバー(smolvla_infer.ipynb)のモデル名は `s2_mix_1025_smolvla`。推論は Colab の GPU サーバー(smolvla_infer.ipynb)を使うので、チェックポイントをコンテナに取り込む必要はない。names.conf の S2_SMOLVLA は `<データ名>_smolvla/final`(`s2_mix_1025_smolvla/final`)。

注意: Colab は、ブラウザのタブを閉じる、無操作が続く、使用時間の上限、のどれかで切れる。セル6の実行中はタブを開いたままにすること。